In [2]:

# STEP 1: DATA ACQUISITION

import pandas as pd
from pathlib import Path

# Dataset Path (Change this to your folder)
DATA_PATH = Path(
    #"/Users/dishitasingh/Desktop/Drexel/INFO442/INFO442-project-main/INFO442-project/Kaggle_dataset"
    "/Users/keith/schoo/netflix_dataset"

)

# Rating Files
rating_files = [
    DATA_PATH / "combined_data_1.txt",
    DATA_PATH / "combined_data_2.txt",
    DATA_PATH / "combined_data_3.txt",
    DATA_PATH / "combined_data_4.txt"
]

print("Checking rating files...\n")

for file in rating_files:
    if file.exists():
        size_gb = file.stat().st_size / (1024**3)
        print(f"✓ {file.name} ({size_gb:.2f} GB)")
    else:
        print(f"✗ {file.name} NOT FOUND")


# Load Movie Titles
movie_file = DATA_PATH / "movie_titles.csv"
movie_records = []
with open(movie_file, "r", encoding="latin-1") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        movie_id, year, title = line.split(",", 2)

        movie_records.append({
            "movie_id": int(movie_id),
            "year": pd.to_numeric(year, errors="coerce"),
            "title": title
        })
movies = pd.DataFrame(movie_records)
movies["year"] = movies["year"].astype("Int64")


# Display Movie Information
print("\nMovie Titles Loaded Successfully!\n")
print(movies.head())

print("\nMovie Dataset Shape:")
print(movies.shape)


Checking rating files...

✓ combined_data_1.txt (0.46 GB)
✓ combined_data_2.txt (0.52 GB)
✓ combined_data_3.txt (0.43 GB)
✓ combined_data_4.txt (0.51 GB)

Movie Titles Loaded Successfully!

   movie_id  year                         title
0         1  2003               Dinosaur Planet
1         2  2004    Isle of Man TT 2004 Review
2         3  1997                     Character
3         4  1994  Paula Abdul's Get Up & Dance
4         5  2004      The Rise and Fall of ECW

Movie Dataset Shape:
(17770, 3)


In [3]:
# STEP 2: Parse and Merge the Four Rating Files
# Output: ratings.csv

output_file = DATA_PATH / "ratings.csv"

# Create output file with header
with open(output_file, "w") as out:
    out.write("user_id,movie_id,rating,date\n")

for file in rating_files:
    print(f"Processing {file.name}...")
    current_movie = None
    rows = []

    with open(file, "r") as f:
        for line in f:
            line = line.strip()

            # Movie ID line
            if line.endswith(":"):
                current_movie = int(line[:-1])

            # Rating line
            else:
                user_id, rating, date = line.split(",")
                rows.append([
                    int(user_id),
                    current_movie,
                    int(rating),
                    date
                ])

            # Save every 1 million rows
            if len(rows) >= 1_000_000:
                pd.DataFrame(
                    rows,
                    columns=["user_id", "movie_id", "rating", "date"]
                ).to_csv(
                    output_file,
                    mode="a",
                    header=False,
                    index=False
                )
                print(f"  Saved {len(rows):,} rows")
                rows = []

    # Save remaining rows
    if rows:
        pd.DataFrame(
            rows,
            columns=["user_id", "movie_id", "rating", "date"]
        ).to_csv(
            output_file,
            mode="a",
            header=False,
            index=False
        )
        print(f"  Saved final {len(rows):,} rows")

print("\nAll rating files merged successfully!")
print(f"Saved to: {output_file}")


Processing combined_data_1.txt...
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved final 53,764 rows
Processing combined_data_2.txt...
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1,000,000 rows
  Saved 1

In [4]:
# Load and clean the movie titles table first (handling commas in titles)
movie_records = []
with open(DATA_PATH / "movie_titles.csv", "r", encoding="latin-1") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Split only on the first two commas to keep commas inside titles intact
        movie_id, year, title = line.split(",", 2)
        movie_records.append({
            "movie_id": int(movie_id),
            "year": pd.to_numeric(year, errors="coerce"),
            "title": title
        })

movies = pd.DataFrame(movie_records)

In [5]:
movies_clean = movies.dropna(subset=['year'])
valid_movie_ids = set(movies_clean['movie_id'])

# 2. Define memory-efficient types
dtypes = {
    'user_id': 'int32',
    'movie_id': 'int32',
    'rating': 'int8'
}

# 3. Read ONLY the columns we need
ratings = pd.read_csv(output_file, dtype=dtypes, usecols=['user_id', 'movie_id', 'rating'])

# 4. Filter the ratings using our clean movie IDs
ratings = ratings[ratings['movie_id'].isin(valid_movie_ids)]

print("Ratings loaded and cleaned successfully.")
print(ratings.shape)

Ratings loaded and cleaned successfully.
(100479542, 3)


In [6]:
# STEP 3: Clean the Data

# 1. Convert data types
ratings["user_id"] = pd.to_numeric(
    ratings["user_id"],
    errors="coerce"
)
ratings["movie_id"] = pd.to_numeric(
    ratings["movie_id"],
    errors="coerce"
)
ratings["rating"] = pd.to_numeric(
    ratings["rating"],
    errors="coerce"
)

# Convert numeric columns to integers after removing nulls
ratings["user_id"] = ratings["user_id"].astype("int32")
ratings["movie_id"] = ratings["movie_id"].astype("int32")
ratings["rating"] = ratings["rating"].astype("int8")

# 4. Validate rating range
invalid_ratings = ~ratings["rating"].isin([1, 2, 3, 4, 5])
print(
    f"\nRows with invalid ratings: "
    f"{invalid_ratings.sum():,}"
)
# Remove invalid ratings if any exist
ratings = ratings[
    ratings["rating"].isin([1, 2, 3, 4, 5])
]

# 5. Validate movie ID range
invalid_movie_ids = ~ratings["movie_id"].between(
    1,
    17770
)
print(
    f"Rows with invalid movie IDs: "
    f"{invalid_movie_ids.sum():,}"
)
# Remove invalid movie IDs if any exist
ratings = ratings[
    ratings["movie_id"].between(1, 17770)
]

# 6. Final verification
print("\nFinal data types:")
print(ratings.dtypes)
print("\nMissing values after cleaning:")
print(ratings.isnull().sum())
print("\nRating values:")
print(sorted(ratings["rating"].unique()))
print(
    "\nMovie ID range:",
    ratings["movie_id"].min(),
    "to",
    ratings["movie_id"].max()
)
print(f"\nFinal number of rows: {len(ratings):,}")
print("\nCleaned data sample:")
print(ratings.head())


Rows with invalid ratings: 0
Rows with invalid movie IDs: 0

Final data types:
user_id     int32
movie_id    int32
rating       int8
dtype: object

Missing values after cleaning:
user_id     0
movie_id    0
rating      0
dtype: int64

Rating values:
[1, 2, 3, 4, 5]

Movie ID range: 1 to 17770

Final number of rows: 100,479,542

Cleaned data sample:
   user_id  movie_id  rating
0  1488844         1       3
1   822109         1       5
2   885013         1       4
3    30878         1       4
4   823519         1       3


In [7]:
# STEP 6: Filter Sparse Data

TOP_USERS = 10000
MIN_MOVIE_RATINGS = 50

# Keep the 10,000 most active users
user_counts = ratings.groupby("user_id").size()

top_users = user_counts.nlargest(TOP_USERS).index

ratings_filtered = ratings[
    ratings["user_id"].isin(top_users)
].copy()

# Keep movies with at least 50 ratings
movie_counts = ratings_filtered.groupby("movie_id").size()

popular_movies = movie_counts[
    movie_counts >= MIN_MOVIE_RATINGS
].index

ratings_filtered = ratings_filtered[
    ratings_filtered["movie_id"].isin(popular_movies)
].copy()

# Summary
print("=" * 50)
print("FILTERED DATASET SUMMARY")
print("=" * 50)

print(f"Original Ratings : {len(ratings):,}")
print(f"Filtered Ratings : {len(ratings_filtered):,}")

print(f"\nOriginal Users : {ratings['user_id'].nunique():,}")
print(f"Filtered Users : {ratings_filtered['user_id'].nunique():,}")

print(f"\nOriginal Movies : {ratings['movie_id'].nunique():,}")
print(f"Filtered Movies : {ratings_filtered['movie_id'].nunique():,}")

print(f"\nData Retained: {(len(ratings_filtered)/len(ratings))*100:.2f}%")

FILTERED DATASET SUMMARY
Original Ratings : 100,479,542
Filtered Ratings : 15,351,759

Original Users : 480,189
Filtered Users : 10,000

Original Movies : 17,763
Filtered Movies : 13,503

Data Retained: 15.28%


In [ ]:

# STEP 7: Build the User–Item Sparse Matrix

from scipy.sparse import csr_matrix

# Create consecutive matrix positions for user IDs and movie IDs
user_ids = ratings_filtered["user_id"].unique()
movie_ids = ratings_filtered["movie_id"].unique()

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}
movie_to_index = {
    movie_id: index
    for index, movie_id in enumerate(movie_ids)
}
index_to_user = {
    index: user_id
    for user_id, index in user_to_index.items()
}
index_to_movie = {
    index: movie_id
    for movie_id, index in movie_to_index.items()
}

# Convert IDs into row and column positions
user_indices = ratings_filtered["user_id"].map(user_to_index)
movie_indices = ratings_filtered["movie_id"].map(movie_to_index)

# Build sparse user-item matrix
# Rows = users
# Columns = movies
# Values = ratings
user_item_matrix = csr_matrix(
    (
        ratings_filtered["rating"].astype("float32"),
        (user_indices, movie_indices)
    ),
    shape=(len(user_ids), len(movie_ids))
)

# Verify the matrix
print("User–Item Matrix Created Successfully")
print(f"Matrix shape       : {user_item_matrix.shape}")
print(f"Number of users    : {user_item_matrix.shape[0]:,}")
print(f"Number of movies   : {user_item_matrix.shape[1]:,}")
print(f"Stored ratings     : {user_item_matrix.nnz:,}")

total_cells = (
    user_item_matrix.shape[0]
    * user_item_matrix.shape[1]
)

density = (
    user_item_matrix.nnz
    / total_cells
) * 100

sparsity = 100 - density

print(f"Matrix density     : {density:.4f}%")
print(f"Matrix sparsity    : {sparsity:.4f}%")

User–Item Matrix Created Successfully
Matrix shape       : (10000, 13503)
Number of users    : 10,000
Number of movies   : 13,503
Stored ratings     : 15,351,759
Matrix density     : 11.3691%
Matrix sparsity    : 88.6309%


In [9]:
#item-user sparce matrix

# Rows = movies
# Columns = users
item_user_matrix = user_item_matrix.T.tocsr()

print("\nItem–User Matrix Shape:")
print(item_user_matrix.shape)


Item–User Matrix Shape:
(13503, 10000)


In [10]:
# STEP 9: Apply Truncated SVD (linear dimentionality reduction)
# applied to user–item matrix

from sklearn.decomposition import TruncatedSVD

N_COMPONENTS = 100

svd = TruncatedSVD(
    n_components=N_COMPONENTS,
    random_state=42
)

movie_features = svd.fit_transform(item_user_matrix)

print("Truncated SVD Completed")
print(f"Original item-user shape : {item_user_matrix.shape}")
print(f"Reduced movie shape      : {movie_features.shape}")

explained_variance = svd.explained_variance_ratio_.sum()

print(f"Number of components     : {N_COMPONENTS}")
print(f"Explained variance       : {explained_variance:.2%}")

Truncated SVD Completed
Original item-user shape : (13503, 10000)
Reduced movie shape      : (13503, 100)
Number of components     : 100
Explained variance       : 57.70%


In [11]:
from sklearn.neighbors import NearestNeighbors

N_NEIGHBORS = 11  # 10 similar movies + itself

knn_model = NearestNeighbors(
    n_neighbors=N_NEIGHBORS,
    metric="cosine",
    algorithm="brute" 
)

knn_model.fit(movie_features)

print("KNN model fitted on SVD movie features")

KNN model fitted on SVD movie features


In [16]:
from scipy.sparse import csr_matrix
import numpy as np

np.random.seed(42)

# For each user, hold out ~20% of their ratings as "test" (set to 0 in train matrix)
def train_test_split_sparse(matrix, test_size=0.2):
    train = matrix.copy().tolil()
    test = csr_matrix(matrix.shape, dtype="float32").tolil()

    for user_idx in range(matrix.shape[0]):
        item_indices = matrix[user_idx].nonzero()[1]
        if len(item_indices) < 5:
            continue  # skip users with too few ratings to split meaningfully

        n_test = max(1, int(len(item_indices) * test_size))
        test_items = np.random.choice(item_indices, size=n_test, replace=False)

        for item_idx in test_items:
            test[user_idx, item_idx] = train[user_idx, item_idx]
            train[user_idx, item_idx] = 0

    return train.tocsr(), test.tocsr()

train_matrix, test_matrix = train_test_split_sparse(user_item_matrix, test_size=0.2)

print(f"Train ratings: {train_matrix.nnz:,}")
print(f"Test ratings : {test_matrix.nnz:,}")

Train ratings: 12,285,415
Test ratings : 3,066,344


In [17]:
from scipy.sparse.linalg import svds

N_FACTORS = 50  # latent dimensions — tune this

# svds returns U, sigma, Vt for the largest k singular values
U, sigma, Vt = svds(train_matrix.astype("float32"), k=N_FACTORS)

# svds returns factors in ascending singular value order — reverse to descending
U = U[:, ::-1]
sigma = sigma[::-1]
Vt = Vt[::-1, :]

sigma_diag = np.diag(sigma)

# Latent factor matrices
user_factors = U @ np.sqrt(sigma_diag)        # shape: (n_users, k)
item_factors = Vt.T @ np.sqrt(sigma_diag)     # shape: (n_movies, k)

print(f"User factors shape : {user_factors.shape}")
print(f"Item factors shape : {item_factors.shape}")

User factors shape : (10000, 50)
Item factors shape : (13503, 50)


In [18]:
from sklearn.neighbors import NearestNeighbors

item_knn = NearestNeighbors(metric="cosine", algorithm="brute")
item_knn.fit(item_factors)

def get_similar_movies(movie_id, k=10):
    """Return the k movies most similar to the given movie_id, via SVD latent space."""
    if movie_id not in movie_to_index:
        raise ValueError(f"movie_id {movie_id} not found")

    movie_idx = movie_to_index[movie_id]
    query = item_factors[movie_idx].reshape(1, -1)

    distances, indices = item_knn.kneighbors(query, n_neighbors=k + 1)

    # Drop the first result — it's the movie itself
    similar = [
        (index_to_movie[idx], 1 - dist)  # convert cosine distance -> similarity
        for idx, dist in zip(indices[0][1:], distances[0][1:])
    ]
    return similar

# Example
example_movie = movie_ids[0]
print(f"Movies similar to movie_id={example_movie}:")
for mid, sim in get_similar_movies(example_movie, k=5):
    print(f"  movie_id={mid}  similarity={sim:.4f}")

Movies similar to movie_id=1:
  movie_id=694  similarity=0.8262
  movie_id=5302  similarity=0.8155
  movie_id=5661  similarity=0.7913
  movie_id=14526  similarity=0.7642
  movie_id=10257  similarity=0.7633


In [26]:
def recommend_for_user_v2(user_id, k_neighbors=20, top_n=10, matrix=train_matrix, min_votes=2):
    user_idx = user_to_index[user_id]
    rated_indices = matrix[user_idx].nonzero()[1]
    rated_ratings = matrix[user_idx, rated_indices].toarray().flatten()

    if len(rated_indices) == 0:
        return []

    user_mean = rated_ratings.mean()

    weighted_dev_sum = np.zeros(item_factors.shape[0], dtype="float32")
    weight_sum = np.zeros(item_factors.shape[0], dtype="float32")
    vote_count = np.zeros(item_factors.shape[0], dtype="int32")

    query = item_factors[rated_indices]
    distances, indices = item_knn.kneighbors(query, n_neighbors=k_neighbors + 1)

    for row, rating in zip(range(len(rated_indices)), rated_ratings):
        neighbor_idx = indices[row][1:]
        neighbor_sim = 1 - distances[row][1:]
        deviation = rating - user_mean

        weighted_dev_sum[neighbor_idx] += neighbor_sim * deviation
        weight_sum[neighbor_idx] += np.abs(neighbor_sim)
        vote_count[neighbor_idx] += 1

    with np.errstate(invalid="ignore", divide="ignore"):
        predicted = user_mean + np.where(weight_sum > 0, weighted_dev_sum / weight_sum, 0)

    # Damp / exclude low-confidence candidates (too few contributing votes)
    predicted[vote_count < min_votes] = -np.inf
    predicted[rated_indices] = -np.inf  # exclude already-rated

    top_indices = np.argpartition(predicted, -top_n)[-top_n:]
    top_indices = top_indices[np.argsort(-predicted[top_indices])]

    return [
        (index_to_movie[idx], predicted[idx], vote_count[idx])
        for idx in top_indices if predicted[idx] > -np.inf
    ]

In [30]:
def precision_recall_at_k(test_matrix, k=10, relevance_threshold=4.0, n_users_sample=None, matrix=train_matrix):
    """
    For each user with test ratings, recommend top-k movies and check
    how many are 'relevant' (test rating >= threshold) and were actually held out.
    """
    precisions = []
    recalls = []

    users_with_test = np.unique(test_matrix.nonzero()[0])

    if n_users_sample is not None:
        users_with_test = np.random.choice(
            users_with_test, size=min(n_users_sample, len(users_with_test)), replace=False
        )

    for user_idx in users_with_test:
        user_id = index_to_user[user_idx]

        test_item_indices = test_matrix[user_idx].nonzero()[1]
        test_ratings = test_matrix[user_idx, test_item_indices].toarray().flatten()

        relevant_items = set(
            index_to_movie[idx] for idx, r in zip(test_item_indices, test_ratings)
            if r >= relevance_threshold
        )

        if len(relevant_items) == 0:
            continue  # nothing relevant in test set for this user, skip

        recs = recommend_for_user(user_id, top_n=k, matrix=matrix)
        recommended_items = set(mid for mid, _ in recs)

        n_relevant_and_recommended = len(recommended_items & relevant_items)

        precision = n_relevant_and_recommended / k
        recall = n_relevant_and_recommended / len(relevant_items)

        precisions.append(precision)
        recalls.append(recall)

    avg_precision = np.mean(precisions) if precisions else 0.0
    avg_recall = np.mean(recalls) if recalls else 0.0

    return avg_precision, avg_recall, len(precisions)

# Run evaluation (sample users for speed on large datasets)
K = 10
precision, recall, n_evaluated = precision_recall_at_k(
    test_matrix, k=K, relevance_threshold=4.0, n_users_sample=500
)

print(f"Evaluated on {n_evaluated} users")
print(f"Precision@{K}: {precision:.4f}")
print(f"Recall@{K}   : {recall:.4f}")
if precision + recall > 0:
    f1 = 2 * precision * recall / (precision + recall)
    print(f"F1@{K}       : {f1:.4f}")

Evaluated on 500 users
Precision@10: 0.1110
Recall@10   : 0.0081
F1@10       : 0.0150


In [31]:
movies

,movie_id,year,title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW
...,...,...,...
17765,17766,2002.0,Where the Wild Things Are and Other Maurice Se...
17766,17767,2004.0,Fidel Castro: American Experience
17767,17768,2000.0,Epoch
17768,17769,2003.0,The Company


In [35]:
movie_id_to_title = dict(zip(movies["movie_id"], movies["title"]))

def explain_recommendations(user_id, k_neighbors=20, top_n=10, matrix=train_matrix):
    user_idx = user_to_index[user_id]
    rated_indices = matrix[user_idx].nonzero()[1]
    rated_ratings = matrix[user_idx, rated_indices].toarray().flatten()
    rated_movie_ids = [index_to_movie[idx] for idx in rated_indices]

    recs = recommend_for_user_v2(user_id, k_neighbors=k_neighbors, top_n=top_n, matrix=matrix)

    print(f"User {user_id} — top {len(recs)} recommendations:\n")
    for movie_id, score, n_votes in recs:
        title = movie_id_to_title.get(movie_id, f"[unknown title: {movie_id}]")
        print(f"→ {title}  (movie_id={movie_id})")
        print(f"   predicted_score={score:.3f}   contributing_votes={n_votes}")

        # Show which of the user's rated movies this recommendation is similar to
        candidate_idx = movie_to_index[movie_id]
        sims = []
        for r_idx, r_movie_id, r_rating in zip(rated_indices, rated_movie_ids, rated_ratings):
            sim = 1 - np.linalg.norm(  # or use cosine directly via item_knn if preferred
                item_factors[r_idx] / np.linalg.norm(item_factors[r_idx])
                - item_factors[candidate_idx] / np.linalg.norm(item_factors[candidate_idx])
            )
            sims.append((r_movie_id, movie_id_to_title.get(r_movie_id, "?"), sim, r_rating))

        sims.sort(key=lambda x: -x[2])
        print("Movies that user rated highly and is similar to:")
        for r_id, r_title, sim, r_rating in sims[:3]:
            print(f"     - {r_title}  (your rating={r_rating:.0f}, similarity={sim:.3f})")
        print()

explain_recommendations(user_ids[0])

User 1488844 — top 10 recommendations:

→ The Lord of the Rings: The Fellowship of the Ring: Extended Edition  (movie_id=7230)
   predicted_score=5.000   contributing_votes=9
Movies that user rated highly and is similar to:
     - Lord of the Rings: The Return of the King: Extended Edition  (your rating=5, similarity=0.851)
     - Lord of the Rings: The Two Towers: Extended Edition  (your rating=5, similarity=0.850)
     - Lord of the Rings: The Return of the King  (your rating=5, similarity=0.495)

→ MacGyver: Season 3  (movie_id=7147)
   predicted_score=5.000   contributing_votes=2
Movies that user rated highly and is similar to:
     - Magnum P.I.: Season 1  (your rating=4, similarity=0.474)
     - Quantum Leap: Season 3  (your rating=5, similarity=0.457)
     - Home Improvement: Season 1  (your rating=3, similarity=0.452)

→ Star Wars: Episode V: The Empire Strikes Back  (movie_id=5582)
   predicted_score=4.927   contributing_votes=12
Movies that user rated highly and is similar to

In [37]:
def evaluate_at_multiple_k(test_matrix, k_values=[5, 10, 20, 50], relevance_threshold=4.0,
                            n_users_sample=500, matrix=train_matrix, k_neighbors=20, min_votes=2):
    """
    Run precision/recall/F1 for several K values in one pass — reuses the same
    recommendation candidates per user rather than recomputing from scratch each time.
    """
    results = {k: {"precision": [], "recall": []} for k in k_values}
    max_k = max(k_values)
    users_with_test = np.unique(test_matrix.nonzero()[0])
    if n_users_sample is not None:
        users_with_test = np.random.choice(
            users_with_test, size=min(n_users_sample, len(users_with_test)), replace=False
        )
    for user_idx in users_with_test:
        user_id = index_to_user[user_idx]
        test_item_indices = test_matrix[user_idx].nonzero()[1]
        test_ratings = test_matrix[user_idx, test_item_indices].toarray().flatten()
        relevant_items = set(
            index_to_movie[idx] for idx, r in zip(test_item_indices, test_ratings)
            if r >= relevance_threshold
        )
        if len(relevant_items) == 0:
            continue

        # Get recommendations once at the largest K, then slice for smaller K
        recs = recommend_for_user_v2(
            user_id, k_neighbors=k_neighbors, top_n=max_k, matrix=matrix, min_votes=min_votes
        )
        recommended_ids_ranked = [mid for mid, _, _ in recs]   # <-- fixed: 3-tuple unpack

        for k in k_values:
            top_k_items = set(recommended_ids_ranked[:k])
            hits = len(top_k_items & relevant_items)
            precision = hits / k
            recall = hits / len(relevant_items)
            results[k]["precision"].append(precision)
            results[k]["recall"].append(recall)

    summary = []
    for k in k_values:
        avg_p = np.mean(results[k]["precision"]) if results[k]["precision"] else 0.0
        avg_r = np.mean(results[k]["recall"]) if results[k]["recall"] else 0.0
        f1 = 2 * avg_p * avg_r / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0.0
        summary.append({"K": k, "precision": avg_p, "recall": avg_r, "f1": f1,
                         "n_users": len(results[k]["precision"])})
    return summary

# Run the sweep
sweep_results = evaluate_at_multiple_k(
    test_matrix, k_values=[5, 10, 20, 50, 100], relevance_threshold=4.0,
    n_users_sample=500, k_neighbors=20, min_votes=2
)

print(f"{'K':>5} | {'Precision':>10} | {'Recall':>10} | {'F1':>10} | {'N Users':>8}")
print("-" * 55)
for row in sweep_results:
    print(f"{row['K']:>5} | {row['precision']:>10.4f} | {row['recall']:>10.4f} | {row['f1']:>10.4f} | {row['n_users']:>8}")

    K |  Precision |     Recall |         F1 |  N Users
-------------------------------------------------------
    5 |     0.1040 |     0.0041 |     0.0078 |      500
   10 |     0.1028 |     0.0078 |     0.0145 |      500
   20 |     0.1074 |     0.0162 |     0.0282 |      500
   50 |     0.1163 |     0.0442 |     0.0641 |      500
  100 |     0.1244 |     0.0938 |     0.1070 |      500


In [23]:
def diagnose_candidate_pool(user_id, k_neighbors=10, matrix=train_matrix):
    user_idx = user_to_index[user_id]
    rated_indices = matrix[user_idx].nonzero()[1]

    scores = np.zeros(item_factors.shape[0], dtype="float32")
    weight_sums = np.zeros(item_factors.shape[0], dtype="float32")

    query = item_factors[rated_indices]
    distances, indices = item_knn.kneighbors(query, n_neighbors=k_neighbors + 1)

    for row in range(len(rated_indices)):
        neighbor_idx = indices[row][1:]
        weight_sums[neighbor_idx] += 1  # just count coverage

    n_candidates_with_signal = (weight_sums > 0).sum()
    print(f"User {user_id}: rated {len(rated_indices)} movies")
    print(f"  Unique candidates with ANY neighbor signal: {n_candidates_with_signal}")
    print(f"  Total catalog size: {item_factors.shape[0]}")

# Check a few sample users
for uid in user_ids[:5]:
    diagnose_candidate_pool(uid)
    print()

User 1488844: rated 1762 movies
  Unique candidates with ANY neighbor signal: 4775
  Total catalog size: 13503

User 30878: rated 1026 movies
  Unique candidates with ANY neighbor signal: 3346
  Total catalog size: 13503

User 1248029: rated 1164 movies
  Unique candidates with ANY neighbor signal: 3568
  Total catalog size: 13503

User 1080361: rated 970 movies
  Unique candidates with ANY neighbor signal: 3101
  Total catalog size: 13503

User 558634: rated 1281 movies
  Unique candidates with ANY neighbor signal: 4111
  Total catalog size: 13503



In [25]:
def diagnose_score_distribution(user_id, k_neighbors=10, top_n=100, matrix=train_matrix):
    recs = recommend_for_user(user_id, k_neighbors=k_neighbors, top_n=top_n, matrix=matrix)
    scores = [s for _, s in recs]

    print(f"User {user_id}: top {len(scores)} scores")
    print(f"  Max score   : {scores[0]:.4f}")
    print(f"  Min score   : {scores[-1]:.4f}")
    print(f"  Score range : {scores[0] - scores[-1]:.4f}")
    print(f"  Std dev     : {np.std(scores):.4f}")
    print(f"  Top 10 scores : {[round(s,3) for s in scores[:10]]}")
    print(f"  Scores 90-100 : {[round(s,3) for s in scores[90:100]]}")

for uid in user_ids[:3]:
    diagnose_score_distribution(uid)
    print()

# Check the distribution of #ratings per user in your eval sample
eval_users_sample = np.random.choice(np.unique(test_matrix.nonzero()[0]), size=500, replace=False)
n_ratings_per_user = [train_matrix[u].nnz for u in eval_users_sample]

print(f"Median ratings/user: {np.median(n_ratings_per_user)}")
print(f"Mean ratings/user  : {np.mean(n_ratings_per_user):.1f}")
print(f"Max ratings/user   : {np.max(n_ratings_per_user)}")

User 1488844: top 100 scores
  Max score   : 5.0000
  Min score   : 4.0000
  Score range : 1.0000
  Std dev     : 0.3755
  Top 10 scores : [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]
  Scores 90-100 : [4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0]

User 30878: top 100 scores
  Max score   : 5.0000
  Min score   : 5.0000
  Score range : 0.0000
  Std dev     : 0.0000
  Top 10 scores : [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]
  Scores 90-100 : [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]

User 1248029: top 100 scores
  Max score   : 5.0000
  Min score   : 5.0000
  Score range : 0.0000
  Std dev     : 0.0000
  Top 10 scores : [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]
  Scores 90-100 : [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]

Median ratings/user: 1087.0
Mean ratings/user  : 1231.2
Max ratings/user   : 10336
